# EXFOR Experiment Search, Filtering, and Data Access

This notebook demonstrates the extended EXFOR module capabilities:
- Discover all available quantities and reactions in the database
- Filter experiments by target, quantity, year, author, energy range
- Support for multiple targets with OR logic
- Load ANY quantity type (CS, DA, FY, DE, etc.) with full data access
- Access EXFOR URLs for entries

**EXFOR Quantity Codes:**
- CS: Cross section data
- DA: Differential data with respect to angle
- FY: Fission product yields
- DE: Differential data with respect to energy
- PY: Product yields
- RI: Resonance integrals
- RP: Resonance parameters
- POL: Polarization data

## 1. Setup and Database Connection

In [1]:
import kika.exfor as exfor
from kika.exfor import X4ProDatabase
from kika.exfor._constants import EXFOR_QUANTITY_CODES, EXFOR_QUANTITY_WILDCARDS, QUANTITY_FAMILIES
import pandas as pd

# Display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

exfor.configure(db_path=r"C:\Users\Usuario\BaradDur\EXFOR\x4sqlite-20251105-c\x4sqlite1.db")

In [2]:
# Connect to the database
# Option 1: Use environment variable KIKA_X4PRO_DB_PATH
# Option 2: Pass path directly
db = X4ProDatabase()

# Show database statistics
stats = db.get_statistics()
print("Database Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value:,}")

Database Statistics:
  total_datasets: 189,313
  total_metadata: 186,939
  angular_distributions: 48,779
  elastic_scattering: 9,888


## 2. Discovering Available Data

### 2.1 List All Unique Quantities

See what types of measurements are available in the database.

In [3]:
# List all unique quantities for neutron-induced reactions
quantities_df = db.list_unique_quantities(projectile="n")
print(f"Found {len(quantities_df)} unique quantity codes\n")
quantities_df.head(20)

Found 26 unique quantity codes



,quantity,description,count
0,CS,Cross section data,37653
1,RP,Resonance parameters,12808
2,DAP,Partial differential data with respect to angle,4376
3,CSP,Partial cross section data,4217
4,DA,Differential data with respect to angle,3909
5,FY,Fission product yields,3676
6,RI,Resonance integrals,2985
7,DAE,Differential data with respect to angle and en...,2271
8,L,Scattering amplitudes,1373
9,SP,Gamma spectra,1366


In [4]:
# Show quantities for a specific target
fe56_quantities = db.list_unique_quantities(projectile="n", target="Fe-56")
print("Quantities available for Fe-56:")
fe56_quantities

Quantities available for Fe-56:


,quantity,description,count
0,CS,Cross section data,252
1,CSP,Partial cross section data,190
2,RP,Resonance parameters,184
3,DAP,Partial differential data with respect to angle,142
4,DA,Differential data with respect to angle,28
5,DE,Differential data with respect to energy,21
6,SP,Gamma spectra,8
7,DAE,Differential data with respect to angle and en...,5
8,L,Scattering amplitudes,4
9,MLT,Unknown/custom,3


### 2.2 List All Unique Reactions

See what reaction types are available.

In [5]:
# List top 30 reactions by count
reactions_df = db.list_unique_reactions(projectile="n")
print(f"Found {len(reactions_df)} unique reaction codes\n")
reactions_df.head(30)

Found 30918 unique reaction codes



,reacode,MT,count
0,"92-U-235(N,F),,SIG",18,200
1,"6-C-0(N,TOT),,SIG",1,157
2,"13-AL-27(N,TOT),,SIG",1,135
3,"1-H-1(N,TOT),,SIG",1,120
4,"82-PB-0(N,TOT),,SIG",1,119
5,"94-PU-239(N,F),,SIG",18,110
6,"26-FE-0(N,TOT),,SIG",1,109
7,"13-AL-27(N,P)12-MG-27,,SIG",103,102
8,"1-H-WTR(N,THS)1-H-WTR,,DA/DE",0,102
9,"1-H-1(N,EL)1-H-1,,DA",2,100


### 2.3 Reference: EXFOR Quantity Codes

The module includes a dictionary of quantity codes with descriptions.

In [6]:
# Show all known quantity code descriptions
print("EXFOR Quantity Code Reference:")
print("="*60)
for code, desc in EXFOR_QUANTITY_CODES.items():
    print(f"  {code:6s} : {desc}")

EXFOR Quantity Code Reference:
  CS     : Cross section data
  CSP    : Partial cross section data
  CST    : Temperature dependent cross section data
  DA     : Differential data with respect to angle
  DAE    : Differential data with respect to angle and energy
  DAP    : Partial differential data with respect to angle
  DE     : Differential data with respect to energy
  DEP    : Partial differential data with respect to energy
  FY     : Fission product yields
  E      : Fission fragment energies
  MFQ    : Miscellaneous fission quantities
  PY     : Product yields
  TT     : Thick target yields
  TTD    : Differential thick target yields
  TTP    : Partial thick target yields
  COR    : Secondary particle correlations
  L      : Scattering amplitudes
  NQ     : Nuclear quantities
  POL    : Polarization data
  RI     : Resonance integrals
  RP     : Resonance parameters
  RR     : Reaction rates
  SP     : Gamma spectra


In [7]:
# Wildcard codes (match multiple types)
print("\nWildcard Quantity Codes:")
print("="*60)
for code, desc in EXFOR_QUANTITY_WILDCARDS.items():
    print(f"  {code:6s} : {desc}")


Wildcard Quantity Codes:
  CS*    : Cross section data (all types: CS, CSP, CST)
  DA*    : Differential data (all types: DA, DAE, DAP)
  TT*    : Thick target yields (all types: TT, TTD, TTP)


In [8]:
# Quantity families for quick filtering
print("\nQuantity Families:")
for family, codes in QUANTITY_FAMILIES.items():
    print(f"  {family}: {codes}")


Quantity Families:
  cross_section: ['CS', 'CSP', 'CST']
  angular: ['DA', 'DAE', 'DAP']
  energy_spectrum: ['DE', 'DEP']
  fission: ['FY', 'E', 'MFQ']
  product_yields: ['PY', 'TT', 'TTD', 'TTP']
  resonance: ['RI', 'RP']
  polarization: ['POL']
  other: ['COR', 'L', 'NQ', 'RR', 'SP']


## 3. Filtering Experiments

### 3.1 Single Target Filter

In [9]:
# Find angular distribution experiments for Fe-56 elastic scattering
# quantity="DA" for differential with respect to angle
fe56_da = db.list_experiments_general(targets="Fe-56", quantity="DA", mt=2)
print(f"Found {len(fe56_da)} Fe-56 elastic scattering DA experiments")
fe56_da.head(10)

Found 18 Fe-56 elastic scattering DA experiments


,dataset_id,author,year,target,quantity,reacode,ndat,energy_min_mev,energy_max_mev
0,10037024,Boschung,1971,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",19,5.050000,5.58000
1,10958004,El-kadi,1982,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",104,7.960000,13.92000
2,11276009,Rodgers,1967,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",5,2.330000,2.33000
3,11708006,Kinney,1968,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",85,4.600000,7.55000
4,12862003,Mellema,1983,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",82,11.000000,26.00000
5,13511004,Perey,1991,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",37362,0.035015,0.85155
6,14462002,Ramirez,2017,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",192,1.300000,7.96000
7,20482005,Cabe,1967,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",63,0.630000,0.97000
8,20482006,Cabe,1967,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA,,LEG",21,0.630000,0.97000
9,22189004,Hata,1990,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",5,14.130000,14.13000


### 3.2 Multiple Targets (OR Logic)

Query multiple targets at once - returns experiments for ANY of the listed targets.

In [10]:
# Get experiments for Fe-56 OR natural iron (Fe-0)
# quantity="CS" for cross section data
iron_experiments = db.list_experiments_general(
    targets=["Fe-56", "Fe-0"],
    quantity="CS"
)
print(f"Found {len(iron_experiments)} cross section experiments for Fe-56 or Fe-nat")
iron_experiments.head(10)

Found 898 cross section experiments for Fe-56 or Fe-nat


,dataset_id,author,year,target,quantity,reacode,ndat,energy_min_mev,energy_max_mev
0,10001003,Hockenbury,1969,Fe-0,CS,"26-FE-0(N,G),,SIG,,RAW",3264,0.002485,0.18600
1,10006002,Schwartz,1974,Fe-0,CS,"26-FE-0(N,TOT),,SIG",3350,0.495400,15.01500
2,10011002,Carlson,1970,Fe-0,CS,"26-FE-0(N,TOT),,SIG",3726,0.462120,9.01566
3,10020003,Robinson,1969,Fe-0,CS,"26-FE-0(N,TOT),,SIG",1,14.970000,14.97000
4,10022010,Barrall,1969,Fe-56,CS,"26-FE-56(N,P)25-MN-56,,SIG",1,14.600000,14.60000
5,10025013,Drake,1970,Fe-0,CSP,"26-FE-0(N,X)0-G-0,PAR,SIG",3,6.000000,7.50000
6,10031005,Barrall,1969,Fe-56,CS,"26-FE-56(N,P)25-MN-56,,SIG",1,14.800000,14.80000
7,10037004,Boschung,1971,Fe-56,CS,"26-FE-56(N,EL)26-FE-56,,SIG",2,5.050000,5.58000
8,10037005,Boschung,1971,Fe-56,CS,"26-FE-56(N,TOT),,SIG",1,5.050000,5.05000
9,10037015,Boschung,1971,Fe-56,CSP,"26-FE-56(N,INL)26-FE-56,PAR,SIG",12,5.050000,5.58000


In [11]:
# Multiple iron isotopes
all_iron = db.list_experiments_general(
    targets=["Fe-54", "Fe-56", "Fe-57", "Fe-58", "Fe-0"],
    quantity="DA",
    mt=2
)
print(f"Found {len(all_iron)} DA experiments across all iron isotopes")
print("\nBreakdown by target:")
print(all_iron.groupby('target').size())

Found 155 DA experiments across all iron isotopes

Breakdown by target:
target
Fe-0     121
Fe-54     16
Fe-56     18
dtype: int64


### 3.3 Year and Author Filters

In [12]:
# Find experiments from year 2000 onwards
# quantity="CS" for cross section
recent_experiments = db.list_experiments_general(
    targets="U-235",
    quantity="CS",
    year_min=2000
)
print(f"Found {len(recent_experiments)} U-235 CS experiments from 2000+")
recent_experiments.head(10)

Found 141 U-235 CS experiments from 2000+


,dataset_id,author,year,target,quantity,reacode,ndat,energy_min_mev,energy_max_mev
0,13786002,Younes,2001,U-235,CSP,"92-U-235(N,F)36-KR-88,PAR,SIG,G",182,1.011,227.975
1,13786003,Younes,2001,U-235,CSP,"92-U-235(N,F)38-SR-94,PAR,SIG,G",91,1.011,227.975
2,13786004,Younes,2001,U-235,CSP,"92-U-235(N,F)40-ZR-96,PAR,SIG,G",91,1.011,227.975
3,13786005,Younes,2001,U-235,CSP,"92-U-235(N,F)38-SR-98,PAR,SIG,G",91,1.011,227.975
4,13786006,Younes,2001,U-235,CSP,"92-U-235(N,F)40-ZR-98,PAR,SIG,G",91,1.011,227.975
5,13786007,Younes,2001,U-235,CSP,"92-U-235(N,F)40-ZR-100,PAR,SIG,G",182,1.011,227.975
6,13786008,Younes,2001,U-235,CSP,"92-U-235(N,F)40-ZR-102,PAR,SIG,G",91,1.011,227.975
7,13786009,Younes,2001,U-235,CSP,"92-U-235(N,F)42-MO-102,PAR,SIG,G",91,1.011,227.975
8,13786010,Younes,2001,U-235,CSP,"92-U-235(N,F)42-MO-106,PAR,SIG,G",182,1.011,227.975
9,13786011,Younes,2001,U-235,CSP,"92-U-235(N,F)44-RU-108,PAR,SIG,G",182,1.011,227.975


In [13]:
# Find experiments by a specific author
# Note: partial match is supported
author_experiments = db.list_experiments_general(
    targets="Fe-56",
    author="Kinney"
)
print(f"Found {len(author_experiments)} Fe-56 experiments by Kinney")
author_experiments

Found 6 Fe-56 experiments by Kinney


,dataset_id,author,year,target,quantity,reacode,ndat,energy_min_mev,energy_max_mev
0,11708002,Kinney,1968,Fe-56,CSP,"26-FE-56(N,INL)26-FE-56,PAR,SIG",30,4.6,7.55
1,11708003,Kinney,1968,Fe-56,CS,"26-FE-56(N,EL)26-FE-56,,SIG",7,4.6,7.55
2,11708004,Kinney,1968,Fe-56,CSP,"26-FE-56(N,INL)26-FE-56,PAR,SIG",7,4.6,7.55
3,11708005,Kinney,1968,Fe-56,CSP,"26-FE-56(N,INL)26-FE-56,PAR,SIG",12,4.6,7.55
4,11708006,Kinney,1968,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",85,4.6,7.55
5,11708007,Kinney,1968,Fe-56,DAP,"26-FE-56(N,INL)26-FE-56,PAR,DA",85,4.6,7.55


### 3.4 Combined Complex Filter

In [14]:
# Complex filter: multiple targets, specific quantity, year range
complex_query = db.list_experiments_general(
    targets=["Fe-56", "Fe-0"],
    quantity="DA",
    mt=2,
    year_min=1970,
    year_max=1990
)
print(f"Found {len(complex_query)} experiments matching complex criteria")
complex_query

Found 63 experiments matching complex criteria


,dataset_id,author,year,target,quantity,reacode,ndat,energy_min_mev,energy_max_mev
0,10037024,Boschung,1971,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",19,5.0500,5.5800
1,10111011,Kinney,1970,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA",168,4.6000,8.5600
2,10332004,Cox,1972,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA",8,0.9021,0.9021
3,10332075,Cox,1972,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA,,LEG",4,0.9021,0.9021
4,10384004,Velkley,1974,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA,,LEG",10,8.9700,8.9700
...,...,...,...,...,...,...,...,...,...
58,40179004,Korzh,1972,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA,,LEG",6,1.5000,1.5000
59,40417003,Tutubalin,1973,Fe-56,DA,"26-FE-56(N,EL)26-FE-56,,DA",16,14.7000,14.7000
60,40532004,Korzh,1977,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA",36,1.5000,3.0000
61,40738003,Gusejnov,1983,Fe-0,DA,"26-FE-0(N,EL)26-FE-0,,DA,,LEG/AV",5,0.6250,0.6250


## 4. Loading and Inspecting Experiments

### 4.1 Load Single Experiment (General Method)

In [15]:
# Load any experiment type using the general method
# First, find an experiment
cs_experiments = db.list_experiments_general(targets="Fe-56", quantity="CS")
if len(cs_experiments) > 0:
    dataset_id = cs_experiments.iloc[0]['dataset_id']
    print(f"Loading experiment: {dataset_id}")

Loading experiment: 10022010


In [16]:
# Load the experiment
if len(cs_experiments) > 0:
    exp = db.load_experiment_general(dataset_id)
    print(exp.summary())

EXFOR Experiment: 10022.010
  Label:       Barrall (1969)
  Quantity:    CS (Cross section data)
  Target:      Fe56 (ZAID: 26056)
  Projectile:  n
  Data points: 1
  Variables:   ['value', 'error', 'energy']
  Units:       {'value': 'B', 'energy': 'EV'}
  URL:         https://www-nds.iaea.org/exfor/10022.010
  energy range: 1.46e+07 - 1.46e+07


### 4.2 Access EXFOR URLs

In [17]:
# The experiment has a URL to the IAEA EXFOR database
if len(cs_experiments) > 0:
    print(f"Dataset ID: {exp.dataset_id}")
    print(f"EXFOR URL:  {exp.exfor_url}")
    print(f"\nClick the URL above to view the full entry on the IAEA website")

Dataset ID: 10022010
EXFOR URL:  https://www-nds.iaea.org/exfor/10022.010

Click the URL above to view the full entry on the IAEA website


### 4.3 Access Metadata

In [18]:
if len(cs_experiments) > 0:
    print("Experiment Metadata:")
    print(f"  Entry:      {exp.entry}")
    print(f"  Subentry:   {exp.subentry}")
    print(f"  Label:      {exp.label}")
    print(f"  Citation:   {exp.citation}")
    print(f"  Target:     {exp.target} (ZAID: {exp.zaid})")
    print(f"  Projectile: {exp.projectile}")
    print(f"  Quantity:   {exp.quantity}")
    print(f"  Description: {exp.quantity_description}")

Experiment Metadata:
  Entry:      10022
  Subentry:   010
  Label:      Barrall (1969)
  Citation:   {'authors': ['Barrall'], 'year': 1969, 'reference': 'EXFOR 10022010'}
  Target:     Fe56 (ZAID: 26056)
  Projectile: n
  Quantity:   CS
  Description: Cross section data


### 4.4 Access Data

In [19]:
if len(cs_experiments) > 0:
    # Get full data as DataFrame
    print(f"Data shape: {exp.data.shape}")
    print(f"Columns: {list(exp.data.columns)}")
    print(f"Units: {exp.units}")
    print(f"\nFirst 10 rows:")
    exp.data.head(10)

Data shape: (1, 3)
Columns: ['value', 'error', 'energy']
Units: {'value': 'B', 'energy': 'EV'}

First 10 rows:


In [20]:
if len(cs_experiments) > 0:
    # Get unique values for independent variables
    print(f"Independent variables: {exp.independent_vars}")
    if 'energy' in exp.data.columns:
        energies = exp.get_unique_values('energy')
        print(f"Unique energies: {len(energies)}")
        print(f"Energy range: {energies.min():.4g} - {energies.max():.4g}")

Independent variables: ['energy']
Unique energies: 1
Energy range: 1.46e+07 - 1.46e+07


In [21]:
if len(cs_experiments) > 0 and 'energy' in exp.data.columns:
    # Filter data by energy range
    e_min = exp.data['energy'].min()
    e_max = e_min * 2
    filtered = exp.filter(energy=(e_min, e_max))
    print(f"Filtered data ({e_min:.4g} - {e_max:.4g}): {len(filtered)} points")
    filtered

Filtered data (1.46e+07 - 2.92e+07): 1 points


## 5. Working with Different Quantity Types

### 5.1 Angular Distributions (DA)

In [22]:
# Find and load a DA experiment
da_experiments = db.list_experiments_general(targets="Fe-56", quantity="DA", mt=2)
if len(da_experiments) > 0:
    da_exp = db.load_experiment_general(da_experiments.iloc[0]['dataset_id'])
    print(da_exp.summary())
    print("\nSample data:")
    da_exp.data.head()

EXFOR Experiment: 10037.024
  Label:       Boschung (1971)
  Quantity:    DA (Differential data with respect to angle)
  Target:      Fe56 (ZAID: 26056)
  Projectile:  n
  Data points: 19
  Variables:   ['value', 'error', 'energy', 'angle']
  Units:       {'value': 'B/SR', 'energy': 'EV', 'angle': 'ADEG'}
  URL:         https://www-nds.iaea.org/exfor/10037.024
  energy range: 5.05e+06 - 5.58e+06
  angle range: 25 - 148

Sample data:


### 5.2 Cross Sections (CS)

In [23]:
# Find and load a CS experiment
cs_experiments2 = db.list_experiments_general(targets="U-235", quantity="CS")
if len(cs_experiments2) > 0:
    cs_exp = db.load_experiment_general(cs_experiments2.iloc[0]['dataset_id'])
    print(cs_exp.summary())
    print("\nSample data:")
    cs_exp.data.head()

EXFOR Experiment: 10013.003
  Label:       Lounsbury (1970)
  Quantity:    CS (Cross section data)
  Target:      U235 (ZAID: 92235)
  Projectile:  n
  Data points: 1
  Variables:   ['value', 'error', 'energy']
  Units:       {'value': 'NO-DIM', 'energy': 'EV'}
  URL:         https://www-nds.iaea.org/exfor/10013.003
  energy range: 0.0253 - 0.0253

Sample data:


### 5.3 Fission Yields (FY)

In [24]:
# Find fission yield experiments
fy_experiments = db.list_experiments_general(targets="U-235", quantity="FY")
print(f"Found {len(fy_experiments)} fission yield experiments for U-235")
if len(fy_experiments) > 0:
    fy_experiments.head()

Found 1215 fission yield experiments for U-235


In [25]:
# Load one if available
if len(fy_experiments) > 0:
    fy_exp = db.load_experiment_general(fy_experiments.iloc[0]['dataset_id'])
    print(fy_exp.summary())
    print("\nSample data:")
    fy_exp.data.head()

EXFOR Experiment: 10517.003
  Label:       Flynn (1975)
  Quantity:    FY (Fission product yields)
  Target:      U235 (ZAID: 92235)
  Projectile:  n
  Data points: 3
  Variables:   ['value', 'error', 'energy', 'product_zam']
  Units:       {'value': 'PART/FIS', 'energy': 'EV', 'product_zam': 'NO-DIM'}
  URL:         https://www-nds.iaea.org/exfor/10517.003
  energy range: 0.0253 - 0.0253
  product_zam range: 3.708e+04 - 5.514e+04

Sample data:


### 5.4 Energy Differential (DE)

In [26]:
# Find energy differential experiments
de_experiments = db.list_experiments_general(targets="U-235", quantity="DE")
print(f"Found {len(de_experiments)} DE experiments for U-235")
if len(de_experiments) > 0:
    de_experiments.head()

Found 26 DE experiments for U-235


In [27]:
# Load one if available
if len(de_experiments) > 0:
    de_exp = db.load_experiment_general(de_experiments.iloc[0]['dataset_id'])
    print(de_exp.summary())
    print("\nSample data:")
    de_exp.data.head()

EXFOR Experiment: 13799.002
  Label:       Sharfuddin (1988)
  Quantity:    DEP (Partial differential data with respect to energy)
  Target:      U235 (ZAID: 92235)
  Projectile:  n
  Data points: 1000
  Variables:   ['value', 'error', 'spe', 'secondary_energy']
  Units:       {'value': 'ARB-UNITS', 'spe': 'EV', 'secondary_energy': 'EV'}
  URL:         https://www-nds.iaea.org/exfor/13799.002
  spe range: 1.8 - 1.8
  secondary_energy range: 1e+04 - 2e+06

Sample data:


### 5.5 Resonance Parameters (RP)

In [28]:
# Find resonance parameter experiments
rp_experiments = db.list_experiments_general(targets="U-235", quantity="RP")
print(f"Found {len(rp_experiments)} RP experiments for U-235")
if len(rp_experiments) > 0:
    rp_experiments.head()

Found 173 RP experiments for U-235


## 6. Practical Examples

### 6.1 Find All Elastic Scattering Data for Iron Isotopes

In [29]:
# Get all elastic scattering angular distributions for iron isotopes
iron_elastic = db.list_experiments_general(
    targets=["Fe-54", "Fe-56", "Fe-57", "Fe-58", "Fe-0"],
    quantity="DA",
    mt=2
)

print(f"Total experiments: {len(iron_elastic)}")
print("\nBreakdown by target:")
print(iron_elastic.groupby('target').agg({'dataset_id': 'count', 'year': ['min', 'max']}))

Total experiments: 155

Breakdown by target:
       dataset_id  year      
            count   min   max
target                       
Fe-0          121  1952  2019
Fe-54          16  1967  2024
Fe-56          18  1967  2017


### 6.2 Compare Experiments from Different Decades

In [30]:
# 1970s experiments
exp_1970s = db.list_experiments_general(
    targets="Fe-56",
    quantity="DA",
    mt=2,
    year_min=1970,
    year_max=1979
)
print(f"1970s experiments: {len(exp_1970s)}")

# 2000s experiments
exp_2000s = db.list_experiments_general(
    targets="Fe-56",
    quantity="DA",
    mt=2,
    year_min=2000,
    year_max=2009
)
print(f"2000s experiments: {len(exp_2000s)}")

1970s experiments: 6
2000s experiments: 2


### 6.3 Export Filtered Results to CSV

In [31]:
# Get experiments and export metadata
fe56_all = db.list_experiments_general(targets="Fe-56")
print(f"Total Fe-56 experiments: {len(fe56_all)}")

# Uncomment to save to CSV:
# fe56_all.to_csv("fe56_experiments.csv", index=False)
# print("Saved to fe56_experiments.csv")

Total Fe-56 experiments: 843


In [32]:
# Export actual data from an experiment
if len(da_experiments) > 0:
    exp_data = da_exp.to_dataframe()
    print(f"Experiment data shape: {exp_data.shape}")
    
    # Uncomment to save:
    # exp_data.to_csv(f"{da_exp.dataset_id}_data.csv", index=False)

Experiment data shape: (19, 4)


## 7. Summary

Key features demonstrated:

1. **Discovery Methods**
   - `db.list_unique_quantities()` - Find all available quantity types
   - `db.list_unique_reactions()` - Find all reaction codes

2. **Quantity Codes** (use these for filtering)
   - `CS` - Cross section data
   - `DA` - Differential data with respect to angle
   - `FY` - Fission product yields
   - `DE` - Differential data with respect to energy
   - `PY` - Product yields
   - `RI` - Resonance integrals
   - `RP` - Resonance parameters
   - See `EXFOR_QUANTITY_CODES` for full list

3. **Flexible Filtering**
   - Multiple targets with OR logic: `targets=["Fe-56", "Fe-0"]`
   - Year ranges: `year_min=2000, year_max=2020`
   - Author search: `author="Kinney"`
   - Quantity types: `quantity="CS"`, `quantity="DA"`, `quantity="FY"`

4. **General Experiment Loading**
   - `db.load_experiment_general()` works with ANY quantity type
   - Returns `ExforExperiment` with full data access

5. **Data Access**
   - `exp.data` - Full DataFrame
   - `exp.filter()` - Filter by variable values
   - `exp.get_unique_values()` - Get unique values for a variable
   - `exp.exfor_url` - Link to IAEA database

6. **Constants**
   - `EXFOR_QUANTITY_CODES` - Descriptions for quantity codes
   - `EXFOR_QUANTITY_WILDCARDS` - Wildcard codes (CS*, DA*, TT*)
   - `QUANTITY_FAMILIES` - Groups of related quantities

In [ ]:
# Clean up
db.close()

: 